In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import glob
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Constants
DATA_DIR = "train"
MAX_INPUT_FRAMES = 50
MAX_OUTPUT_FRAMES = 25
FIELD_X, FIELD_Y = 120.0, 53.3
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

print(f"Using device: {device}")

# Load and preprocess training data
def load_tracking_data(data_dir):
    input_files = sorted(glob.glob(os.path.join(data_dir, "input_2023_w*.csv")))
    output_files = sorted(glob.glob(os.path.join(data_dir, "output_2023_w*.csv")))
    df_input = pd.concat((pd.read_csv(f) for f in input_files), ignore_index=True)
    df_output = pd.concat((pd.read_csv(f) for f in output_files), ignore_index=True)
    return df_input, df_output

df_input, df_output = load_tracking_data(DATA_DIR)
print(f"Loaded {len(df_input)} input records, {len(df_output)} output records")

# Preprocess data
drop_cols = ["player_name", "player_height", "player_weight", "player_birth_date"]
df_input.drop(columns=drop_cols, inplace=True, errors="ignore")

# Store original play_direction for flipping
play_directions = df_input[['game_id', 'play_id', 'play_direction']].drop_duplicates()

# Encode categorical variables
cat_cols = ["player_position", "player_role", "player_side", "play_direction"]
encoders = {col: LabelEncoder().fit(df_input[col].astype(str)) for col in cat_cols}

# FIX 1: Check the actual encoding mapping
print("\n=== FIX 1: Verify play_direction encoding ===")
direction_mapping = dict(zip(encoders['play_direction'].classes_, 
                             encoders['play_direction'].transform(encoders['play_direction'].classes_)))
print(f"Direction encoding: {direction_mapping}")

for col, le in encoders.items():
    df_input[col] = le.transform(df_input[col].astype(str))

# FIX 2: Store normalization constants for reuse on test data
print("\n=== FIX 2: Store normalization constants ===")
s_max = df_input["s"].max()
a_max = df_input["a"].max()
print(f"Speed max: {s_max:.2f}, Acceleration max: {a_max:.2f}")

# FIX 3: Correct coordinate flipping with direction and velocity
def flip_coordinates_complete(df, flip_cols=['x', 'ball_land_x'], flip_angles=['dir', 'o']):
    """Flip coordinates AND angles for plays going right-to-left"""
    df_flipped = df.copy()
    
    # Determine which plays to flip (right direction)
    # After alphabetical encoding: 'left'=0, 'right'=1
    right_to_left_mask = df['play_direction'] == direction_mapping.get('right', 1)
    
    print(f"Flipping {right_to_left_mask.sum()} / {len(df)} records (right-to-left plays)")
    
    # Flip x-coordinates
    for col in flip_cols:
        if col in df_flipped.columns:
            df_flipped.loc[right_to_left_mask, col] = FIELD_X - df_flipped.loc[right_to_left_mask, col]
    
    # Flip angles (direction and orientation)
    for col in flip_angles:
        if col in df_flipped.columns:
            df_flipped.loc[right_to_left_mask, col] = (180 - df_flipped.loc[right_to_left_mask, col]) % 360
    
    return df_flipped, right_to_left_mask

print("\n=== FIX 3: Apply complete coordinate flipping ===")
df_input, input_flip_mask = flip_coordinates_complete(df_input, ['x', 'ball_land_x'], ['dir', 'o'])

# Apply coordinate flipping to output data
df_output_merged = df_output.merge(play_directions, on=['game_id', 'play_id'], how='left')
df_output_merged['play_direction'] = encoders['play_direction'].transform(df_output_merged['play_direction'].astype(str))
df_output, output_flip_mask = flip_coordinates_complete(df_output_merged, ['x'], [])

# Normalize features
df_input["x_norm"] = df_input["x"] / FIELD_X
df_input["y_norm"] = df_input["y"] / FIELD_Y
df_input["s_norm"] = df_input["s"] / s_max
df_input["a_norm"] = df_input["a"] / a_max
df_output["x"] = df_output["x"] / FIELD_X
df_output["y"] = df_output["y"] / FIELD_Y

# FIX 4: Compute velocity AFTER flipping (now directions are correct)
df_input["dir_rad"] = np.deg2rad(df_input["dir"])
df_input["vx"] = df_input["s_norm"] * np.cos(df_input["dir_rad"])
df_input["vy"] = df_input["s_norm"] * np.sin(df_input["dir_rad"])

# Ball-relative features
df_input["x_rel_to_ball"] = (df_input["x"] - df_input["ball_land_x"]) / FIELD_X
df_input["y_rel_to_ball"] = (df_input["y"] - df_input["ball_land_y"]) / FIELD_Y
df_input["dist_to_ball"] = np.sqrt(df_input["x_rel_to_ball"]**2 + df_input["y_rel_to_ball"]**2)

# FIX 5: Add absolute ball landing positions to features
df_input["ball_land_x_norm"] = df_input["ball_land_x"] / FIELD_X
df_input["ball_land_y_norm"] = df_input["ball_land_y"] / FIELD_Y

feature_cols = ["x_norm", "y_norm", "vx", "vy", "a_norm", "player_position", 
                "player_role", "player_side", "play_direction", 
                "ball_land_x_norm", "ball_land_y_norm",  # Added
                "x_rel_to_ball", "y_rel_to_ball", "dist_to_ball"]

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
print("Data preprocessing completed with corrected flipping")

# FIX 6: Consistent player ordering by nfl_id
def prepare_play_data(play_df, output_df, game_id, play_id, feature_cols):
    """Prepare input and output sequences with consistent player ordering"""
    frames_in = play_df.sort_values(["frame_id", "nfl_id"])  # Sort by both
    
    # Get unique players and their order
    unique_nfl_ids = sorted(play_df["nfl_id"].unique())
    nfl_id_to_idx = {nfl_id: idx for idx, nfl_id in enumerate(unique_nfl_ids)}
    
    # Input frames
    grouped_frames = []
    for frame_id, frame_df in frames_in.groupby("frame_id"):
        frame_df = frame_df.sort_values("nfl_id")  # Ensure consistent ordering
        mat = frame_df[feature_cols].to_numpy()
        if mat.shape[0] < 22:
            mat = np.pad(mat, ((0, 22 - mat.shape[0]), (0, 0)), mode="constant")
        grouped_frames.append(mat)
    
    if not grouped_frames:
        return None, None, None
        
    X_seq = np.stack(grouped_frames)
    if X_seq.shape[0] < MAX_INPUT_FRAMES:
        X_seq = np.pad(X_seq, ((0, MAX_INPUT_FRAMES - X_seq.shape[0]), (0, 0), (0, 0)), mode="constant")
    else:
        X_seq = X_seq[:MAX_INPUT_FRAMES]

    # Output frames
    play_out = output_df[(output_df["game_id"] == game_id) & (output_df["play_id"] == play_id)]
    play_out = play_out.sort_values(["frame_id", "nfl_id"])
    
    frames_out = []
    for frame_id, frame_df in play_out.groupby("frame_id"):
        frame_df = frame_df.sort_values("nfl_id")
        mat = frame_df[["x", "y"]].to_numpy()
        if mat.shape[0] < 22:
            mat = np.pad(mat, ((0, 22 - mat.shape[0]), (0, 0)), mode="constant")
        frames_out.append(mat)
    
    if not frames_out:
        return None, None, None
        
    Y_seq = np.stack(frames_out)
    if Y_seq.shape[0] < MAX_OUTPUT_FRAMES:
        Y_seq = np.pad(Y_seq, ((0, MAX_OUTPUT_FRAMES - Y_seq.shape[0]), (0, 0), (0, 0)), mode="constant")
    else:
        Y_seq = Y_seq[:MAX_OUTPUT_FRAMES]

    # FIX 7: Mask with consistent ordering
    mask = np.zeros(22, dtype=bool)
    for idx, nfl_id in enumerate(unique_nfl_ids):
        player_data = play_df[play_df["nfl_id"] == nfl_id]
        if player_data["player_to_predict"].max():
            mask[idx] = True

    return X_seq, Y_seq, mask

# Prepare training data
print("\n=== FIX 6 & 7: Prepare data with consistent ordering ===")
plays = df_input.groupby(["game_id", "play_id"])
X_data, Y_data, masks = [], [], []

for (game_id, play_id), play_df in plays:
    X_seq, Y_seq, mask = prepare_play_data(play_df, df_output, game_id, play_id, feature_cols)
    if X_seq is not None:
        X_data.append(X_seq)
        Y_data.append(Y_seq)
        masks.append(mask)

X_data = np.array(X_data)
Y_data = np.array(Y_data)
masks = np.array(masks)

print(f"Training data shapes: X={X_data.shape}, Y={Y_data.shape}, masks={masks.shape}")

# FIX 8: Create validation split
print("\n=== FIX 8: Create validation split ===")
n_plays = len(X_data)
train_idx, val_idx = train_test_split(range(n_plays), test_size=0.15, random_state=42)
X_train, X_val = X_data[train_idx], X_data[val_idx]
Y_train, Y_val = Y_data[train_idx], Y_data[val_idx]
masks_train, masks_val = masks[train_idx], masks[val_idx]

print(f"Train: {len(X_train)} plays, Val: {len(X_val)} plays")

# Improved Seq-to-Seq Graph Neural Network
class FrameGraphModule(nn.Module):
    def __init__(self, feat_dim, hidden_dim):
        super().__init__()
        self.intra_team_fc = nn.Linear(feat_dim, hidden_dim)
        self.bipartite_fc = nn.Linear(feat_dim, hidden_dim)
        self.to_landing_fc = nn.Linear(feat_dim, hidden_dim)
        self.out_fc = nn.Linear(hidden_dim * 3, feat_dim)

    def forward(self, x, adj_intra, adj_bipartite, adj_to_landing):
        intra_msg = torch.matmul(adj_intra, self.intra_team_fc(x))
        bip_msg = torch.matmul(adj_bipartite, self.bipartite_fc(x))
        landing_msg = torch.matmul(adj_to_landing, self.to_landing_fc(x))
        combined = torch.cat([intra_msg, bip_msg, landing_msg], dim=-1)
        return self.out_fc(combined) + x

# FIX 9: Improved autoregressive decoder
class Seq2SeqGraphTransformer(nn.Module):
    def __init__(self, feat_dim, hidden_dim, n_heads=4, n_layers=2, seq_out=25):
        super().__init__()
        self.frame_graph = FrameGraphModule(feat_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=22*feat_dim, nhead=n_heads, batch_first=True)
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.seq_out = seq_out
        self.feat_dim = feat_dim
        
        # Improved decoder
        self.output_proj = nn.Linear(22*feat_dim, 22*2)
        self.state_update = nn.GRUCell(22*2, 22*feat_dim)  # Better than simple linear

    def forward(self, x, adj_intra_seq, adj_bipartite_seq, adj_to_landing_seq):
        batch, seq_in, num_players, feat_dim = x.shape
        
        # Vectorized frame-wise graph updates
        x_bt = x.view(batch * seq_in, num_players, feat_dim)
        adj_intra_bt = adj_intra_seq.view(batch * seq_in, num_players, num_players)
        adj_bipartite_bt = adj_bipartite_seq.view(batch * seq_in, num_players, num_players)
        adj_to_landing_bt = adj_to_landing_seq.view(batch * seq_in, num_players, num_players)

        # Graph operations
        intra_msg = torch.matmul(adj_intra_bt, self.frame_graph.intra_team_fc(x_bt))
        bip_msg = torch.matmul(adj_bipartite_bt, self.frame_graph.bipartite_fc(x_bt))
        landing_msg = torch.matmul(adj_to_landing_bt, self.frame_graph.to_landing_fc(x_bt))

        combined = torch.cat([intra_msg, bip_msg, landing_msg], dim=-1)
        x_bt_out = self.frame_graph.out_fc(combined) + x_bt

        # Reshape and temporal encoding
        x_seq = x_bt_out.view(batch, seq_in, num_players * feat_dim)
        x_seq = self.temporal_encoder(x_seq)

        # Improved autoregressive prediction with GRU
        preds = []
        hidden_state = x_seq[:, -1, :]
        
        for t in range(self.seq_out):
            out = self.output_proj(hidden_state)
            pred_frame = out.view(batch, num_players, 2)
            preds.append(pred_frame)
            
            # Update hidden state using GRU
            hidden_state = self.state_update(out, hidden_state)
            
        return torch.stack(preds, dim=1)

# Create model
feat_dim = X_data.shape[-1]
hidden_dim = 64
graph_model = Seq2SeqGraphTransformer(feat_dim=feat_dim, hidden_dim=hidden_dim, seq_out=MAX_OUTPUT_FRAMES)
graph_model.to(device)
optimizer = torch.optim.Adam(graph_model.parameters(), lr=5e-4)

print(f"\n=== FIX 9: Improved model architecture ===")
print(f"Graph model created with {sum(p.numel() for p in graph_model.parameters()):,} parameters")

# FIX 10: Compute adjacency on-the-fly during training
def compute_adjacency_batch(play_dfs, batch_indices, df_input_full):
    """Compute adjacency matrices for a batch of plays"""
    batch_size = len(batch_indices)
    adj_intra_seq = np.zeros((batch_size, MAX_INPUT_FRAMES, 22, 22))
    adj_bipartite_seq = np.zeros_like(adj_intra_seq)
    adj_to_landing_seq = np.zeros_like(adj_intra_seq)
    
    plays = list(df_input_full.groupby(["game_id", "play_id"]))
    
    for batch_idx, play_idx in enumerate(batch_indices):
        (game_id, play_id), play_df = plays[play_idx]
        frames_in = play_df.sort_values(["frame_id", "nfl_id"])
        
        for frame_idx, (frame_id, frame_df) in enumerate(frames_in.groupby("frame_id")):
            if frame_idx >= MAX_INPUT_FRAMES:
                break
            
            frame_df = frame_df.sort_values("nfl_id")
            player_sides = frame_df['player_side'].values
            present = min(len(player_sides), 22)
            
            for i in range(present):
                for j in range(present):
                    if i == j:
                        continue
                    if player_sides[i] == player_sides[j]:
                        adj_intra_seq[batch_idx, frame_idx, i, j] = 1
                    else:
                        adj_bipartite_seq[batch_idx, frame_idx, i, j] = 1
                    adj_to_landing_seq[batch_idx, frame_idx, i, j] = 1
    
    return (
        torch.tensor(adj_intra_seq, dtype=torch.float32),
        torch.tensor(adj_bipartite_seq, dtype=torch.float32),
        torch.tensor(adj_to_landing_seq, dtype=torch.float32),
    )

# Training with validation
def masked_rmse(pred, target, mask):
    mask = mask.unsqueeze(1).expand(-1, pred.shape[1], -1).unsqueeze(-1)
    diff = (pred - target) ** 2
    masked_diff = diff * mask.float()
    mse = masked_diff.sum() / (mask.sum().float() + 1e-8)
    return torch.sqrt(mse)

print("\n=== Training with validation ===")
best_val_loss = float('inf')
patience = 3
patience_counter = 0

for epoch in range(10):
    # Training
    graph_model.train()
    total_train_loss = 0
    batch_size = 16
    n_train_batches = 0
    
    train_indices = np.random.permutation(len(X_train))
    
    for i in range(0, len(X_train), batch_size):
        batch_end = min(i + batch_size, len(X_train))
        batch_idx = train_indices[i:batch_end]
        
        X_batch = torch.tensor(X_train[batch_idx], dtype=torch.float32, device=device)
        Y_batch = torch.tensor(Y_train[batch_idx], dtype=torch.float32, device=device)
        mask_batch = torch.tensor(masks_train[batch_idx], dtype=torch.bool, device=device)
        
        # Compute adjacency on-the-fly
        adj_intra, adj_bipartite, adj_to_landing = compute_adjacency_batch(
            None, [train_idx[idx] for idx in batch_idx], df_input
        )
        adj_intra = adj_intra.to(device)
        adj_bipartite = adj_bipartite.to(device)
        adj_to_landing = adj_to_landing.to(device)
        
        pred = graph_model(X_batch, adj_intra, adj_bipartite, adj_to_landing)
        loss = masked_rmse(pred, Y_batch, mask_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(graph_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        n_train_batches += 1
    
    avg_train_loss = total_train_loss / n_train_batches
    
    # Validation
    graph_model.eval()
    total_val_loss = 0
    n_val_batches = 0
    
    with torch.no_grad():
        for i in range(0, len(X_val), batch_size):
            batch_end = min(i + batch_size, len(X_val))
            batch_idx = range(i, batch_end)
            
            X_batch = torch.tensor(X_val[list(batch_idx)], dtype=torch.float32, device=device)
            Y_batch = torch.tensor(Y_val[list(batch_idx)], dtype=torch.float32, device=device)
            mask_batch = torch.tensor(masks_val[list(batch_idx)], dtype=torch.bool, device=device)
            
            adj_intra, adj_bipartite, adj_to_landing = compute_adjacency_batch(
                None, [val_idx[idx] for idx in batch_idx], df_input
            )
            adj_intra = adj_intra.to(device)
            adj_bipartite = adj_bipartite.to(device)
            adj_to_landing = adj_to_landing.to(device)
            
            pred = graph_model(X_batch, adj_intra, adj_bipartite, adj_to_landing)
            loss = masked_rmse(pred, Y_batch, mask_batch)
            total_val_loss += loss.item()
            n_val_batches += 1
    
    avg_val_loss = total_val_loss / n_val_batches
    
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(graph_model.state_dict(), 'best_model.pth')
        print(f"  → Saved best model (val_loss: {best_val_loss:.5f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping triggered")
            break

# Load best model
graph_model.load_state_dict(torch.load('best_model.pth'))
print("\nTraining completed! Loaded best model for inference.")

# FIX 11: Corrected test data processing
print("\n=== FIX 11: Process test data correctly ===")
test_input = pd.read_csv("test_input.csv")
test_submission_format = pd.read_csv("sample_submission.csv")

test_input_processed = test_input.copy()
test_input_processed.drop(columns=drop_cols, inplace=True, errors="ignore")

# Apply same encoding
for col in cat_cols:
    if col in test_input_processed.columns:
        test_input_processed[col] = encoders[col].transform(test_input_processed[col].astype(str))

# Apply coordinate flipping
test_input_processed, test_flip_mask = flip_coordinates_complete(
    test_input_processed, ['x', 'ball_land_x'], ['dir', 'o']
)

# Use stored normalization constants
test_input_processed["x_norm"] = test_input_processed["x"] / FIELD_X
test_input_processed["y_norm"] = test_input_processed["y"] / FIELD_Y
test_input_processed["s_norm"] = test_input_processed["s"] / s_max  # Use stored max
test_input_processed["a_norm"] = test_input_processed["a"] / a_max  # Use stored max
test_input_processed["dir_rad"] = np.deg2rad(test_input_processed["dir"])
test_input_processed["vx"] = test_input_processed["s_norm"] * np.cos(test_input_processed["dir_rad"])
test_input_processed["vy"] = test_input_processed["s_norm"] * np.sin(test_input_processed["dir_rad"])
test_input_processed["ball_land_x_norm"] = test_input_processed["ball_land_x"] / FIELD_X
test_input_processed["ball_land_y_norm"] = test_input_processed["ball_land_y"] / FIELD_Y
test_input_processed["x_rel_to_ball"] = (test_input_processed["x"] - test